In [ ]:
import torch
import sys
sys.path.append('../')
import time
from ttpi import TTPI 
from pushing_dyn_explicit_double import pusher_slider_sys
torch.set_default_dtype(torch.float64)
%load_ext autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
p_target = torch.tensor([0., 0., 0.]).view(1,-1).to(device)
dt=0.025

dyn_system =  pusher_slider_sys(p_target=p_target,dt=dt, device=device)
dyn_system_test =  pusher_slider_sys(p_target=p_target,dt=dt, device=device)

In [ ]:
# dim=5
w = 0.5
state_max_c = torch.tensor([w,w,torch.pi, -0.065, 0.06]).to(device) # (s_x, s_y, s_theta, p_x, p_y) 
state_min_c =  torch.tensor([-w,-w,-torch.pi, -0.1, -0.06]).to(device)

state_max = torch.tensor([w,w,torch.pi, -0.065, 0.06, 3]).to(device) # (s_x, s_y, s_theta, p_x, p_y, face_id) 
state_min =  torch.tensor([-w,-w,-torch.pi, -0.1, -0.06, 0]).to(device)
is_state_c = torch.tensor([1]*len(state_max)).to(device)
is_state_c[-1] = 0

n = 100
n_state = [n]*5
n_state[2] = 100
n_action = 50

v = 0.1
action_max_c = torch.tensor([v, v]).to(device) # (p_ddx, p_ddy)
action_min_c = torch.tensor([0, -v]).to(device)
action_max =  torch.tensor([v,v,3]).to(device)
action_min = torch.tensor([0,-v,0]).to(device)

In [ ]:
domain_state_c = []
for i in range(len(state_max_c)):#x,dx
    x_n = torch.linspace(state_min_c[i],0,int(n_state[i]/2)).to(device)
    x_p = torch.linspace(0,state_max_c[i],int(n_state[i]/2)).to(device)[1:]
    domain_state_c.append(torch.concat((x_n,x_p),dim=-1))
domain_state = domain_state_c  +[torch.tensor([0.,1.,2.,3.]).to(device)]
domain_action = [torch.tensor([0.,1.,2.,3.]).to(device)]+ [torch.linspace(action_min_c[i],action_max_c[i],n_action).to(device) for i in range(len(action_max_c))]

In [ ]:
def forward_model(state,action):
    next_state = dyn_system.dynamics(state,action)
    return next_state


def reward(state,action):
    rewards = -1*dyn_system.cost_func(state,action,scale=w)
    return rewards

In [ ]:
n_test = 100
dim_state = len(domain_state)
init_state = torch.empty((n_test,dim_state))
for i in range(dim_state):
    init_state[:,i] = state_min[i] + torch.rand(n_test).clip(0.25,0.75).to(device)*(state_max[i]-state_min[i])
state = init_state.to(device)


In [ ]:
tol = torch.tensor([0.03, 0.03, 15/180*torch.pi]).to(device)[:3]

def callback(ttpi, state=state, file_name='fig',callback_count=0):
    print("Testing....")
    
    history = []
    T=2000
    traj = state[:,:].clone()[:,None,:] #bsx(T+1)x6
    traj_actions = torch.empty(state.shape[0],T,3).to(device) #bsxTx3
    cum_reward = torch.tensor([0.]*state.shape[0]).to(device)
    dt_cum = 0
    for i in range(T):
        t0 = time.time()
        
        action = ttpi.policy(state)
        # print(action[0,0])
        t1=time.time()
        dt_cum+=(t1-t0)
        r = ttpi.reward_normalized(state,action)
        cum_reward+=r
        state = forward_model(state,action)
        traj = torch.concat((traj,state[:,None,:]),dim=1)
        traj_actions[:,i,:]=action
    print("time taken by policy: ", dt_cum/T)
    succ_rate = torch.sum(torch.all(torch.abs(state[:,:3])<=tol, dim=1))/n_test
    print(f"Success rate of {n_test} tests is {succ_rate*100}%")
    print(traj_actions[0,:,0])
    from plot_utils import plot_planarpush

    plot_num = 99 # number of plotted tasks
    plt=plot_planarpush(traj[-plot_num:].to('cpu').numpy(),
                        traj_actions[-plot_num:].to('cpu').numpy(), 
                        animation=False, step_skip=40, 
                        xmax=w,x_target=p_target[0].to('cpu').numpy(),figsize=5, 
                        save_as=None,
                        scale=10)
    plt.show()        
    return r.mean().to("cpu"), cum_reward.mean().to("cpu")
    

In [ ]:
ttpi = TTPI(domain_state=domain_state, 
                domain_action=domain_action, 
                reward=reward, 
                normalize_reward=False,
                forward_model=forward_model, 
                gamma=0.99, 
                rmax_v=100, rmax_a=100, 
                nswp_v=5, nswp_a=5, 
                kickrank_v=10, kickrank_a=20,
                max_batch_v=10**4,max_batch_a=10**5,
                eps_cross_v=1e-3,
                eps_cross_a=1e-3,
                eps_round_v=1e-4, 
                eps_round_a=1e-3, 
                n_samples=10, 
                verbose=True, 
                device=device) # action = 'deterministic_tt', 'stochastic_tt', 'random'

In [ ]:

resume= False
ttpi.train(n_iter_max=100,n_iter_v=1,
        callback=callback, callback_freq=10,
        verbose=False, file_name='pushing')


"